# PC-JEPA — Predictive Coding + JEPA on Moving MNIST

**Before running:** `Runtime → Change runtime type → TPU v2` (free tier) or TPU v3/v4.

Architecture : CNN encoder → PC hierarchy (Loop 1, inference) → Transformer predictor → JEPA loss (Loop 2, learning).

> **TPU note** — JAX acquires an exclusive lock on the TPU for the kernel process.
> All JAX-heavy cells use `%run` (IPython magic) instead of `!python` so they
> execute **in the current kernel process** and share the existing TPU lock.
> `!python` would spawn a new process that cannot access the locked TPU.

Cells in order:
1. Install JAX TPU + verify devices
2. Clone repo + install deps
3. Download MNIST
4. Sanity checks
5. `check_tconv` — convergence diagnostic
6. Exp2 — sample efficiency curve (PC-JEPA vs Transformer)

In [ ]:
# ── 1. Install JAX with TPU support ──────────────────────────────────────────
# Must run before any jax import. Colab's default jax may not have libtpu.
!pip install -q "jax[tpu]>=0.4.26" \
    -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

In [ ]:
# ── 2. Verify TPU is visible to JAX ──────────────────────────────────────────
import jax

print(f"JAX version : {jax.__version__}")
print(f"Backend     : {jax.default_backend()}")
print(f"Devices     : {jax.devices()}")

assert jax.default_backend() == "tpu", (
    "TPU not detected. Go to Runtime → Change runtime type → TPU, "
    "then Runtime → Restart and run all."
)

In [ ]:
# ── 3. Clone repo ─────────────────────────────────────────────────────────────
import os

REPO_URL    = "https://github.com/Baptistecaille/JEPA-PC.git"
BRANCH      = "claude/no-ema-sigreg-OHc9i"   # change to 'main' for stable release
REPO_DIR    = "JEPA-PC"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists — pulling latest changes")
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
# ── 4. Install remaining dependencies ─────────────────────────────────────────
# jax[tpu] already installed; skip jax/jaxlib from requirements if present.
!pip install -q optax numpy matplotlib tqdm

In [ ]:
# ── 5. Pre-download MNIST ─────────────────────────────────────────────────────
# The data loader reads from ~/.keras/datasets/mnist.npz.
# tensorflow.keras is pre-installed on Colab and handles the download.
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print(f"MNIST loaded — train: {x_train.shape}, test: {x_test.shape}")

In [ ]:
# ── 6. Sanity checks — all modules ────────────────────────────────────────────
# %run executes in the current kernel process (shares the TPU lock).
# !python would spawn a new process → "TPU already in use" error.
%run run_all.py --mode sanity

In [ ]:
# ── 7. T_conv diagnostic ──────────────────────────────────────────────────────
# Expected: T_conv < 500, MSE final < pc_tol=0.01, 100% des batches convergent.
%run check_tconv.py

In [ ]:
# ── 8. Exp2 — sample efficiency (PC-JEPA vs Transformer) ──────────────────────
# n ∈ (100, 500, 1000, 2000, 5000, 10000) × 2 seeds × 2 models = 24 runs.
# On TPU v2 (8 cores): ~30–60 min depending on n=10000.
%run run_all.py --mode exp2

In [ ]:
# ── 9. Plot results ────────────────────────────────────────────────────────────
import json, glob, os
import numpy as np
import matplotlib.pyplot as plt

# Load most recent exp2 result file
files = sorted(glob.glob("results/exp2_*.json"))
assert files, "No exp2 results found — run cell 8 first."
with open(files[-1]) as f:
    results = json.load(f)
print(f"Loaded: {files[-1]}")

NS     = (100, 500, 1000, 2000, 5000, 10000)
SEEDS  = (42, 137)

pc_means, pc_stds = [], []
tr_means, tr_stds = [], []

for n in NS:
    pc_vals = [results.get(f"pc_jepa_n{n}_seed{s}", {}).get("nmse", float("nan")) for s in SEEDS]
    tr_vals = [results.get(f"transformer_n{n}_seed{s}", {}).get("nmse", float("nan")) for s in SEEDS]
    pc_means.append(np.nanmean(pc_vals))
    pc_stds.append(np.nanstd(pc_vals))
    tr_means.append(np.nanmean(tr_vals))
    tr_stds.append(np.nanstd(tr_vals))

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(NS, pc_means, yerr=pc_stds, marker="o", label="PC-JEPA (ours)", capsize=3)
ax.errorbar(NS, tr_means, yerr=tr_stds, marker="s", linestyle="--",
            label="Transformer baseline", capsize=3)
ax.set_xscale("log")
ax.set_xlabel("Training samples n")
ax.set_ylabel("NMSE (↓)")
ax.set_title("Sample efficiency — PC-JEPA vs Transformer (Moving MNIST)")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("results/sample_efficiency.png", dpi=150)
plt.show()
print("Saved: results/sample_efficiency.png")